# Engine v2 — Andoma

**AI idea:** Minimax search with alpha-beta pruning · **Era:** 1950s–60s · **Week 4** · **Source:** [github.com/healeycodes/andoma](https://github.com/healeycodes/andoma) · **License:** MIT

## 0 · Setup — the same cell in every engine notebook

Installs python-chess if needed, finds Stockfish, and defines the toolkit every notebook uses: `stockfish()`, `show()`, `play_game()`, `play_match()`, `random_bot()`, `material()`, `greedy_bot()`, `save_pgn()`, `save_results()`. The comment block at the top of the cell says what each one takes and returns. If the last line says Stockfish was **not found**, paste its path into `STOCKFISH_PATH` and re-run.

In [ ]:
# ---------------------------------------------------------------- shared setup (same in every engine notebook)
#
#  THE TOOLKIT -- what this cell leaves behind. The same nine functions and four names are
#  in every engine notebook of the ladder, so what you learn here works in v2 through v8.
#  (In a cell, run  help(play_match)  to see any of these again without scrolling back.)
#
#  NAMES YOU CAN READ
#    IN_COLAB        True on Google Colab. When True this cell also apt-installs Stockfish.
#    STOCKFISH_PATH  Empty by default -- paste your own path here if the finder below fails.
#    STOCKFISH       Full path to the Stockfish binary, or None. Printed on the last line.
#    VALUES          Piece values in centipawns: pawn 100, knight 300, bishop 300,
#                    rook 500, queen 900, king 0 (the king is never captured, so it scores 0).
#
#  OPEN AN ENGINE, SEE A BOARD
#    stockfish(elo=None, threads=1)
#                    Start a Stockfish process and return it. Pass elo=1350 to weaken it to
#                    a club player. ALWAYS call .quit() on what it returns, or the process
#                    stays alive after the cell ends.
#    show(board, size=320)
#                    Draw the board in the notebook with the last move highlighted.
#                    Displays; returns nothing.
#
#  PLAY GAMES BETWEEN TWO BOTS
#    A "bot" here is any function that takes a board and returns a legal move --
#    greedy_bot, random_bot, and the turing_bot you write in Section 5 all qualify.
#    play_game(white_fn, black_fn, max_plies=160, start=None)
#                    Play one game; returns the finished board. start takes a FEN if you
#                    want to begin somewhere other than move 1.
#    play_match(fn_a, fn_b, games=10, max_plies=160, seed=1)
#                    Play a match with colours alternating; returns (points_for_a, decisive)
#                    where a win is 1 point, a draw 0.5, and decisive counts games that
#                    actually finished in a result. This is how you measure a change.
#
#  BOTS AND SCORING -- your opponents and your yardstick
#    random_bot(board)
#                    Plays a legal move at random. The floor: anything you write should beat it.
#    material(board, color)
#                    Centipawn material balance from color's point of view (+ is good for color).
#    greedy_bot(board)
#                    The Setup-Week baseline, three rules: play mate if you see it, else take
#                    the most material, else choose at random among equals. Scores about
#                    26-28 out of 30 against random_bot (it moves with the seed) --
#                    beat THIS and you have built something.
#
#  SAVE YOUR EVIDENCE -- both write into the portfolio layout when it exists, else here
#    save_pgn(board, name, white="?", black="?")
#                    Write a finished game to games/<name>; returns the path it used.
#    save_results(rows, header, engine_folder)
#                    Write engines/<engine_folder>/results.csv; returns the path it used.
#                    rows is a list of lists, header a list of column names.
#
# ----------------------------------------------------------------------------------------
%pip install chess -q
import sys, os, shutil, csv, time, random, platform
import chess, chess.engine, chess.pgn, chess.svg
from IPython.display import display, SVG, clear_output

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "-q", "stockfish"], capture_output=True)

STOCKFISH_PATH = ""      # laptop users: paste the full path here if the finder below cannot see it
STOCKFISH = next((p for p in [STOCKFISH_PATH, os.environ.get("STOCKFISH_PATH"), shutil.which("stockfish"),
                              "/usr/games/stockfish", "/usr/local/bin/stockfish", "/opt/homebrew/bin/stockfish",
                              "./stockfish", "./stockfish.exe"] if p and os.path.isfile(p)), None)

def stockfish(elo=None, threads=1):
    """Open a Stockfish process (optionally rating-limited). Remember to .quit() it."""
    if not STOCKFISH:
        raise FileNotFoundError("Stockfish not found -- set STOCKFISH_PATH (see Setup Week Kit, Step 5.1)")
    eng = chess.engine.SimpleEngine.popen_uci(STOCKFISH)
    if elo is not None:
        opt = eng.options["UCI_Elo"]
        eng.configure({"UCI_LimitStrength": True, "UCI_Elo": max(opt.min, min(opt.max, elo))})
    return eng

def show(board, size=320):
    """Draw `board` in the notebook, highlighting the last move. Displays; returns nothing."""
    display(SVG(chess.svg.board(board, size=size, lastmove=board.peek() if board.move_stack else None)))

def play_game(white_fn, black_fn, max_plies=160, start=None):
    """Two Python functions (board -> move) play a game. Returns the final board."""
    b = chess.Board(start) if start else chess.Board()
    while not b.is_game_over() and len(b.move_stack) < max_plies:
        b.push((white_fn if b.turn == chess.WHITE else black_fn)(b))
    return b

def play_match(fn_a, fn_b, games=10, max_plies=160, seed=1):
    """fn_a vs fn_b with colours alternating. Returns (points for A, decisive games)."""
    pts, decisive = 0.0, 0
    for i in range(games):
        random.seed(seed + i)
        a_white = (i % 2 == 0)
        b = play_game(fn_a, fn_b, max_plies) if a_white else play_game(fn_b, fn_a, max_plies)
        res = b.result() if b.is_game_over() else "1/2-1/2"
        if res != "1/2-1/2" and b.is_game_over(): decisive += 1
        if res == "1-0":   pts += 1 if a_white else 0
        elif res == "0-1": pts += 0 if a_white else 1
        else:              pts += 0.5
    return pts, decisive

def random_bot(board):
    """A legal move chosen at random. The weakest possible opponent -- your baseline's baseline."""
    return random.choice(list(board.legal_moves))

VALUES = {chess.PAWN: 100, chess.KNIGHT: 300, chess.BISHOP: 300, chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 0}
def material(board, color):
    """Material balance in centipawns from `color`'s point of view: positive means `color` is ahead."""
    return sum(VALUES[p.piece_type] * (1 if p.color == color else -1) for p in board.piece_map().values())

def greedy_bot(board):
    """The Setup-Week baseline: mate if you see it, else grab the most material, random tie-break."""
    me, best, moves = board.turn, None, []
    for m in board.legal_moves:
        board.push(m); s = 10**6 if board.is_checkmate() else material(board, me); board.pop()
        if best is None or s > best: best, moves = s, [m]
        elif s == best: moves.append(m)
    return random.choice(moves)

def save_pgn(board, name, white="?", black="?"):
    """Save a finished game as PGN into games/ (portfolio layout) if that folder exists, else here."""
    game = chess.pgn.Game.from_board(board)
    game.headers["Event"] = "Gambit Lab"; game.headers["White"] = white; game.headers["Black"] = black
    game.headers["Result"] = board.result() if board.is_game_over() else "*"
    path = os.path.join("games" if os.path.isdir("games") else ".", name)
    with open(path, "w") as f: print(game, file=f, end="\n\n")
    print("saved game ->", path)
    return path

def save_results(rows, header, engine_folder):
    """Write results.csv into engines/<folder>/ if that folder exists (portfolio layout), else here."""
    folder = os.path.join("engines", engine_folder) if os.path.isdir(os.path.join("engines", engine_folder)) else "."
    path = os.path.join(folder, "results.csv")
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)
    print(f"saved {len(rows)} rows -> {path}")
    return path

print("python-chess", chess.__version__, "| Stockfish:", STOCKFISH or "NOT FOUND (only the engine cells need it)")

## 1 · Read first (20 minutes)

<strong>Minimax</strong> is look-ahead with a pessimistic assumption: I pick the move that is best <em>even after</em> the opponent picks their best reply, and so on to a fixed depth. <strong>Alpha-beta pruning</strong> is the observation that once a branch is already worse than something you have, you can stop exploring it — same answer, a fraction of the work.

Andoma is a readable Python engine on top of python-chess with exactly these pieces: alpha-beta search, move ordering (captures first), and a simplified evaluation with piece-square tables. It speaks UCI through <code>main.py</code>.

**The shape of it:** <code>negamax(board, depth, alpha, beta)</code> → a score &nbsp;·&nbsp; count the nodes with pruning on and off &nbsp;·&nbsp; same answer, fewer nodes

### Why this idea, and why now

Turing's bot looked one move ahead, so it happily grabbed a pawn that lost a queen a move later. Minimax fixes that by imagining the opponent's replies (and your replies to those) down to some depth, scoring only the leaves, and passing the scores back up: at your turns you take the maximum, at theirs the minimum. Writing it so every level takes the maximum of the <em>negated</em> child scores is the negamax trick — one function instead of two.

The cost is exponential: about 30 moves per position means depth 3 is 27,000 leaves and depth 4 is 810,000. Alpha-beta keeps two bounds — the best you are guaranteed (alpha) and the best the opponent will allow (beta) — and cuts any branch where they cross. With good move ordering it roughly squares the reachable depth for the same work. This week you measure that yourself.

### Reading list — in this order

1. [Building My Own Chess Engine (Andrew Healey)](https://healeycodes.com/building-my-own-chess-engine) — The author's own walkthrough of Andoma; read it before the code.
2. Russell, S. & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach*, 4th ed., Pearson. Chapter "Adversarial Search and Games" (Ch. 5; Ch. 6 in the Global Edition), section "Optimal Decisions in Games" — the minimax search algorithm, alpha–beta pruning, and move ordering, with the pseudocode to hold beside `movegeneration.py`.
3. Knuth, D. E. & Moore, R. W. (1975). "An Analysis of Alpha-Beta Pruning." *Artificial Intelligence* 6(4), 293–326. doi:10.1016/0004-3702(75)90019-3 — the paper that proved pruning never changes the answer. Read the opening sections, where the negamax procedure is defined and alpha–beta is derived from it; skip the analysis.
4. [Andoma README](https://github.com/healeycodes/andoma) — How it is organised: `main.py` (UCI), `movegeneration.py` (the search — find `minimax`, `alpha`, `beta`), `evaluate.py` (piece values and piece-square tables).

## 2 · Run the engine

Open the repository in a browser: [https://github.com/healeycodes/andoma](https://github.com/healeycodes/andoma). Find the function that calls itself (`minimax` in `movegeneration.py`) and the two names it uses for the bounds. Find how moves are ordered before they are searched (`get_ordered_moves`). Find which file speaks UCI (`main.py`) and which is the terminal UI (`ui.py`).

Get it from GitHub
```bash
git clone https://github.com/healeycodes/andoma.git
```
You already have python-chess from Setup Week — do **not** run `pip install -r requirements.txt`; it pins an old python-chess (1.6.1) and would downgrade yours.

In [ ]:
# Optional: run the clone from here (handy on Colab, where there is no terminal). Set CLONE = True and run.
CLONE = True
if CLONE:
    import subprocess
    for cmd in [
    "git clone https://github.com/healeycodes/andoma.git",
    ]:
        print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-600:])
else:
    print("CLONE is False -- run the commands in a terminal, or set CLONE = True here.")

**Terminal — play it through the harness**

```bash
python scripts/first_engine_game.py --engine "python andoma/main.py --depth 3"
```

**Python — or open it directly** (Andoma must be started *inside* its own folder, hence `cwd`; `--depth` is how deep it searches — it ignores the clock)

```python
import chess.engine
engine = chess.engine.SimpleEngine.popen_uci([sys.executable, "main.py", "--depth", "3"], cwd="andoma")
print(engine.id)          # {'name': 'Andoma', 'author': ...} -- your proof it answered
```

Task: Open Andoma, play a full game against an opponent you choose, print every move as it is played, then show the final board and write the game to `games/` as a PGN. Reuse your Week 3 cell — only the launch line changes.

Write a cell for each opponent:

1. Against you
2. Against Stockfish

## 3 · Make Changes and Measure

Write the search by hand: `negamax` (look ahead to a fixed depth, no pruning, with a node counter) and `negamax_ab` (the same with alpha and beta, and the cut-off). Run both on the same positions — the answer must be identical, the node counts must not be. Every cell runs without the real engine installed.

Plain negamax first — no pruning — with a counter so we can see the work. The evaluation is just material from the side to move's point of view; checkmate and stalemate are handled at the leaves.

In [ ]:
def eval_stm(board):
    """Material from the side-to-move's view; terminal positions handled."""
    if board.is_checkmate(): return -100000
    if board.is_stalemate() or board.is_insufficient_material(): return 0
    return material(board, board.turn)

def negamax(board, depth, counter):
    """Look ahead `depth` plies, assume the opponent answers with their best reply. counter[0] counts nodes."""
    counter[0] += 1
    if depth == 0 or board.is_game_over():
        return eval_stm(board)
    best = -10**9
    for m in board.legal_moves:
        board.push(m); v = -negamax(board, depth - 1, counter); board.pop()
        best = max(best, v)
    return best

c = [0]
print("start position, depth 3, plain negamax:", negamax(chess.Board(), 3, c), "-- nodes visited:", f"{c[0]:,}")

Now the same function with alpha and beta. Only two lines change: the bounds are passed down (swapped and negated, because the opponent's best is my worst), and the loop breaks the moment alpha reaches beta. Same answer, far fewer nodes. Move ordering — captures first — makes the cut-offs come sooner, so it is a switch you can turn off to see what it is worth.

In [ ]:
def negamax_ab(board, depth, alpha, beta, counter, order=True):
    """negamax with alpha-beta pruning. order=True searches captures first (better pruning)."""
    counter[0] += 1
    if depth == 0 or board.is_game_over():
        return eval_stm(board)
    moves = list(board.legal_moves)
    if order:                                            # captures first = better pruning
        moves.sort(key=lambda m: board.is_capture(m), reverse=True)
    best = -10**9
    for m in moves:
        board.push(m); v = -negamax_ab(board, depth - 1, -beta, -alpha, counter, order); board.pop()
        best = max(best, v); alpha = max(alpha, v)
        if alpha >= beta: break                          # the cut-off
    return best

positions = {"start":       chess.Board(),
             "Italian":     chess.Board("r1bq1rk1/ppp2ppp/2np1n2/2b1p3/2B1P3/2PP1N2/PP3PPP/RNBQ1RK1 w - - 0 1"),
             "open middle": chess.Board("r2qkb1r/ppp2ppp/2n2n2/3pp3/4P1b1/3P1N2/PPP1BPPP/RNBQK2R w KQkq - 0 1")}
rows = []
for name, bd in positions.items():
    c1, c2, c3 = [0], [0], [0]
    v1 = negamax(bd, 3, c1)
    v2 = negamax_ab(bd, 3, -10**9, 10**9, c2, order=False)
    v3 = negamax_ab(bd, 3, -10**9, 10**9, c3, order=True)
    assert v1 == v2 == v3, "pruning must never change the answer"
    rows.append([name, 3, c1[0], c2[0], c3[0], round(c1[0] / c3[0], 1)])
    print(f"{name:<12} depth 3: plain {c1[0]:>7,}   alpha-beta {c2[0]:>6,}   +ordering {c3[0]:>6,}   ({c1[0]/c3[0]:.1f}x fewer)")

Make at least one change to the search, while keeping everything else identical. For example:

1. Search depth 2 → 3 (the cell below): does one more ply actually win games, and what does it cost?

2. Move ordering on → off: how many more nodes does the same depth take?

3. Evaluation: material only → material plus a piece-square table (Sunfish's tables are on next week's shelf)

Test it: the node table above, plus a match of 6 games between the two versions with colours alternating. Save both in `results.csv`.

Write in RECIPE.md: what you changed and why that one; the results pasted; one sentence on the node ratio you saw and one on the match, claiming no more than six games can support.

In [ ]:
def ab_bot(depth):
    """Wrap the search in a bot (board -> move): try every root move, keep the best."""
    def bot(board):
        best, best_v, c = None, -10**9, [0]
        moves = list(board.legal_moves); random.shuffle(moves)          # random tie-break
        for m in moves:
            board.push(m); v = -negamax_ab(board, depth - 1, -10**9, 10**9, c); board.pop()
            if v > best_v: best, best_v = m, v
        return best
    return bot

b = chess.Board()
print("depth-3 bot's first move:", b.san(ab_bot(3)(b)))
pts, dec = play_match(ab_bot(3), ab_bot(2), games=6, max_plies=80)                   # <-- THE ONE CHANGE: depth 2 -> 3
print(f"depth 3 vs depth 2: {pts}/6 ({dec} decisive)")
rows.append(["match depth3 vs depth2 (6 games)", "", "", "", "", pts])
save_results(rows, ["position", "depth", "nodes_without_pruning", "nodes_with_pruning",
                    "nodes_with_pruning_and_ordering", "ratio_or_points"], "v2-andoma")